In [62]:
import torch
import matplotlib.pyplot as plt
import re
from dataclasses import dataclass
import math

### Load data

In [63]:
with open('TinyStories.txt', 'r', encoding="utf-8") as f:
    text = f.read()

In [ ]:
print(text[:100])

In [ ]:
len(text)

In [ ]:
tok_training_text = text[:8000000]
model_training_text = text[:6000000]

len(tok_training_text), len(model_training_text)

### BPE algorithm helpers

In [67]:


def text_2_ids(text, vocab):
    ids = [vocab[c] for c in text]
    return ids


def get_pairs_count_from_chunks(all_chunk_ids):
    pairs_count = {}

    for chunk_ids in all_chunk_ids:
        for i in range(len(chunk_ids) - 1):
            curr_pair = (chunk_ids[i], chunk_ids[i + 1])
            pairs_count[curr_pair] = pairs_count.get(curr_pair, 0) + 1

    pc = sorted([(pair, count) for pair, count in pairs_count.items()], reverse=True, key=lambda x: x[1])
    return pc


def merge_pair(pair, ids, new_id):
    if len(ids) < 2:
        return ids

    new_ids = []
    i = 0

    while i < len(ids):
        if i < len(ids) - 1:
            curr_pair = (ids[i], ids[i + 1])
            if curr_pair == pair:
                new_ids.append(new_id)
                i += 2
            else:
                new_ids.append(ids[i])
                i += 1
        else:
            new_ids.append(ids[i])
            i += 1

    return new_ids


def update_vocab(vocab, inv_vocab, pair, new_id):
    new_token = inv_vocab[pair[0]] + inv_vocab[pair[1]]
    vocab[new_token] = new_id
    inv_vocab[new_id] = new_token
    return vocab, inv_vocab


def split_text(text, pattern):
    chunks = re.findall(pattern, text, re.IGNORECASE)
    return chunks


def decode(ids, inv_vocab):
    text = "".join(inv_vocab[id] for id in ids)
    return text


def encode(text, merges, vocab, initial_vocab_size, pattern):
    chunks = split_text(text, pattern)
    all_chunk_ids = []
    for chunk in chunks:
        chunk_ids = text_2_ids(chunk, vocab)
        all_chunk_ids.append(chunk_ids)

    for merge_id in range(initial_vocab_size, len(vocab)):
        if merge_id not in merges:
            continue

        curr_merge = merges[merge_id]
        new_all_chunk_ids = []
        for chunk_ids in all_chunk_ids:
            new_chunk_ids = merge_pair(curr_merge, chunk_ids, merge_id)
            new_all_chunk_ids.append(new_chunk_ids)
        all_chunk_ids = new_all_chunk_ids

    ids = [id for chunk_ids in all_chunk_ids for id in chunk_ids]
    return ids



### Initial vocab

In [ ]:
unique_chars = sorted(set(tok_training_text))

vocab = {c : i for i, c in enumerate(unique_chars)}
inv_vocab = {i : c for i, c in enumerate(unique_chars)}

len(vocab), "".join(unique_chars)

### Tokenizer training loop

In [69]:

merges = {}
initial_vocab_size = len(vocab)
voc_size = len(vocab) # 92
pattern = r"'s|'t|'re|'ve|'m|'ll|'d|'n't|\w+|[^\w\s]+|\s+"

chunks = split_text(tok_training_text, pattern)
all_chunk_ids = []
for chunk in chunks:
    chunk_ids = text_2_ids(chunk, vocab)
    all_chunk_ids.append(chunk_ids)

while len(vocab) < voc_size:

    pairs_count = get_pairs_count_from_chunks(all_chunk_ids)
    if not pairs_count:
        break
    pair = pairs_count[0][0]

    new_all_chunk_ids = []
    for chunk_ids in all_chunk_ids:
        new_chunk_ids = merge_pair(pair, chunk_ids, len(vocab))
        new_all_chunk_ids.append(new_chunk_ids)
    all_chunk_ids = new_all_chunk_ids

    merges[len(vocab)] = pair
    vocab, inv_vocab = update_vocab(vocab, inv_vocab, pair, len(vocab))

    if len(vocab) % 50 == 0:
        print(f"Current vocab size: {len(vocab)}")

### Train / val split

In [ ]:
data = torch.tensor(encode(model_training_text, merges, vocab, initial_vocab_size, pattern), dtype=torch.long)

data.shape, data[:100]


In [ ]:
x = int(0.9 * len(data))

train_data = data[:x]
val_data = data[x:]

len(train_data), len(val_data)

# No classes

In [72]:
# vocab_size = voc_size
# num_emb_dims = 6
# context_length = 3

# batch_size = 4 

# num_heads = 3
# head_size = num_emb_dims // num_heads
# hidden_mul = 4
# num_layers = 1

# dropout = 0.1


# rng = torch.Generator().manual_seed(39)

# att_mask = torch.tril(torch.ones(context_length, context_length))
# att1_dropmask = (torch.rand(batch_size, context_length, context_length, generator=rng) > dropout) * 1/(1-dropout)
# att2_dropmask = (torch.rand(batch_size, context_length, context_length, generator=rng) > dropout) * 1/(1-dropout)
# att3_dropmask = (torch.rand(batch_size, context_length, context_length, generator=rng) > dropout) * 1/(1-dropout)
# mhsa_dropmask = (torch.rand(batch_size, context_length, num_emb_dims, generator=rng) > dropout) * 1/(1-dropout)
# mlp_dropmask = (torch.rand(batch_size, context_length, num_emb_dims * hidden_mul, generator=rng) > dropout) * 1/(1-dropout)

# #            Model params (weights)         #
# # ----------------------------------------- #

# # tok emb params
# TOK = torch.randn((vocab_size, num_emb_dims), generator=rng) * 0.1/num_emb_dims**0.5
# # pos emb params
# POS = torch.randn((context_length, num_emb_dims), generator=rng) * 0.1/num_emb_dims**0.5

# #-------- decoder block params --------#
# # layer norm 1 params 
# gamma1 = torch.randn((num_emb_dims, ), generator=rng) * 0.01 + 1.0
# beta1 = torch.randn((num_emb_dims, ), generator=rng) * 0.01

# #------- mhsa params ---------#
# #--- shsa 1 params ---#
# Wq1 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# Wk1 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# Wv1 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# #---------------------#

# #--- shsa 2 params ---#
# Wq2 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# Wk2 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# Wv2 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# #---------------------#

# #--- shsa 3 params ---#
# Wq3 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# Wk3 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# Wv3 = torch.randn((head_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(head_size**0.5)
# #---------------------#

# # linear layer 1 params
# W1 = torch.randn((num_emb_dims, num_emb_dims), generator=rng) * 2**0.5 * 1/(num_emb_dims**0.5)
# b1 = torch.randn(num_emb_dims, generator=rng) * 0.01
# #-----------------------------#

# # layer norm 2 params 
# gamma2 = torch.randn((num_emb_dims, ), generator=rng) * 0.01 + 1.0
# beta2 = torch.randn((num_emb_dims, ), generator=rng) * 0.01

# #------- mlp params ----------#
# # linear layer 2 params
# W2 = torch.randn((num_emb_dims * hidden_mul, num_emb_dims), generator=rng) * 2**0.5 * 1/(num_emb_dims**0.5)
# b2 = torch.randn(num_emb_dims * hidden_mul, generator=rng) * 0.01

# # linear layer 3 params
# W3 = torch.randn((num_emb_dims, num_emb_dims * hidden_mul), generator=rng) * 2**0.5 * 1/(num_emb_dims**0.5)
# b3 = torch.randn(num_emb_dims, generator=rng) * 0.01
# #-----------------------------#
# #--------------------------------------#

# # layer norm 3 params 
# gamma3 = torch.randn((num_emb_dims, ), generator=rng) * 0.01 + 1.0
# beta3 = torch.randn((num_emb_dims, ), generator=rng) * 0.01

# # linear layer 4 params
# W4 = torch.randn((vocab_size, num_emb_dims), generator=rng) * 2**0.5 * 1/(num_emb_dims**0.5)
# b4 = torch.randn(vocab_size, generator=rng) * 0.01


# # ------------------  all model params  ------------------#
# parameters = [TOK, POS, gamma1, beta1, Wq1, Wk1, Wv1, Wq2, Wk2, Wv2, Wq3, Wk3, Wv3, \
#               W1, b1, gamma2, beta2, W2, b2, W3, b3, gamma3, beta3, W4, b4]
# for p in parameters:
#     p.requires_grad=True
# print("params count:", sum(p.numel() for p in parameters))
# # --------------------------------------------------------#

# # -------------------- dataloader ------------------------#

# def get_batch(split):
#     if split == "train":
#         data = train_data
#     else:
#         data = val_data
#     ni = torch.randint(len(data) - context_length, (batch_size, ), generator=rng)
#     x = torch.stack([data[n : n + context_length] for n in ni])
#     y = torch.stack([data[n + 1 : n + 1 + context_length] for n in ni])
#     return x, y

# xb, yb = get_batch("train")
# print("--------------")
# print("xb:\n", xb)
# print()
# print("yb:\n", yb)

In [73]:

# #--------------------- forward pass ------------------------#

# # tok emb + pos emb
# emb = TOK[xb] + POS 
# # ln1
# ln1meani = emb.sum(dim=-1, keepdims=True) * 1/num_emb_dims
# ln1diff = emb - ln1meani
# ln1diffsq = ln1diff**2
# ln1var = ln1diffsq.sum(dim=-1, keepdims=True) * 1/(num_emb_dims - 1)
# ln1var_sqinv = (ln1var + 1e-5)**-0.5
# ln1x_hat = ln1diff * ln1var_sqinv
# ln1_done = ln1x_hat * gamma1 + beta1
# # shsa1
# q1 = ln1_done @ Wq1.transpose(-2, -1)
# k1 = ln1_done @ Wk1.transpose(-2, -1)
# dotp_att1 = q1 @ k1.transpose(-2, -1)
# scaled_dotp1 = dotp_att1 * 1/head_size**0.5
# masked_att1 = scaled_dotp1.masked_fill(att_mask == 0, float("-inf"))
# masked_att1_shifted = masked_att1 - masked_att1.max(dim=-1, keepdims=True).values
# exp_att1 = masked_att1_shifted.exp()
# sum_exp_att1 = exp_att1.sum(dim=-1, keepdims=True)
# inv_sum_exp_att1 = sum_exp_att1**-1
# att1 = exp_att1 * inv_sum_exp_att1
# att1_drop = att1 * att1_dropmask
# v1 = ln1_done @ Wv1.transpose(-2, -1)
# att1_done = att1_drop @ v1
# # shsa2
# q2 = ln1_done @ Wq2.transpose(-2, -1)
# k2 = ln1_done @ Wk2.transpose(-2, -1)
# dotp_att2 = q2 @ k2.transpose(-2, -1)
# scaled_dotp2 = dotp_att2 * 1/head_size**0.5
# masked_att2 = scaled_dotp2.masked_fill(att_mask == 0, float("-inf"))
# masked_att2_shifted = masked_att2 - masked_att2.max(dim=-1, keepdims=True).values
# exp_att2 = masked_att2_shifted.exp()
# sum_exp_att2 = exp_att2.sum(dim=-1, keepdims=True)
# inv_sum_exp_att2 = sum_exp_att2**-1
# att2 = exp_att2 * inv_sum_exp_att2
# att2_drop = att2 * att2_dropmask
# v2 = ln1_done @ Wv2.transpose(-2, -1)
# att2_done = att2_drop @ v2
# # shsa3
# q3 = ln1_done @ Wq3.transpose(-2, -1)
# k3 = ln1_done @ Wk3.transpose(-2, -1)
# dotp_att3 = q3 @ k3.transpose(-2, -1)
# scaled_dotp3 = dotp_att3 * 1/head_size**0.5
# masked_att3 = scaled_dotp3.masked_fill(att_mask == 0, float("-inf"))
# masked_att3_shifted = masked_att3 - masked_att3.max(dim=-1, keepdims=True).values
# exp_att3 = masked_att3_shifted.exp()
# sum_exp_att3 = exp_att3.sum(dim=-1, keepdims=True)
# inv_sum_exp_att3 = sum_exp_att3**-1
# att3 = exp_att3 * inv_sum_exp_att3
# att3_drop = att3 * att3_dropmask
# v3 = ln1_done @ Wv3.transpose(-2, -1)
# att3_done = att3_drop @ v3
# # mhsa
# cat_attheads = torch.cat([att1_done, att2_done, att3_done], dim=-1)
# mhsa_proj = cat_attheads @ W1.transpose(-2, -1) + b1
# mhsa_done = mhsa_proj * mhsa_dropmask
# # res conn
# resconn_mhsa_done = emb + mhsa_done
# # ln2
# ln2meani = resconn_mhsa_done.sum(dim=-1, keepdims=True) * 1/num_emb_dims
# ln2diff = resconn_mhsa_done - ln2meani
# ln2diffsq = ln2diff**2
# ln2var = ln2diffsq.sum(dim=-1, keepdims=True) * 1/(num_emb_dims - 1)
# ln2var_sqinv = (ln2var + 1e-5)**-0.5
# ln2x_hat = ln2diff * ln2var_sqinv
# ln2_done = ln2x_hat * gamma2 + beta2
# # mlp
# mlp_lin1 = ln2_done @ W2.transpose(-2, -1) + b2
# mlp_relu = torch.maximum(mlp_lin1, torch.tensor(0))
# mlp_drop = mlp_relu * mlp_dropmask
# mlp_lin2 = mlp_drop @ W3.transpose(-2, -1) + b3
# # res conn
# resconn_mlp_done = resconn_mhsa_done + mlp_lin2
# # ln3
# ln3meani = resconn_mlp_done.sum(dim=-1, keepdims=True) * 1/num_emb_dims
# ln3diff = resconn_mlp_done - ln3meani 
# ln3diffsq = ln3diff**2
# ln3var = ln3diffsq.sum(dim=-1, keepdims=True) * 1/(num_emb_dims - 1)
# ln3var_sqinv = (ln3var + 1e-5)**-0.5
# ln3x_hat = ln3diff * ln3var_sqinv
# ln3_done = ln3x_hat * gamma3 + beta3
# # model head (output layer)
# logits = ln3_done @ W4.transpose(-2, -1) + b4
# # flatten
# B, T, C = logits.shape
# logits_flat = logits.view(B * T, C)
# targets = yb.view(B * T)
# # softmax
# logits_flat_shifted = logits_flat - logits_flat.max(dim=-1, keepdims=True).values
# logits_exp = logits_flat_shifted.exp()
# sum_exp_logits = logits_exp.sum(dim=-1, keepdims=True)
# inv_sum_exp_logits = sum_exp_logits**-1
# probs = logits_exp * inv_sum_exp_logits
# # cross entropy loss
# probs_log = probs.log()
# loss = -probs_log[torch.arange(len(targets)), targets].mean()


# #--------------------- pytorch backward pass ------------------------#
# for p in parameters:
#     p.grad = None
    
# _tensors = [probs_log, probs, inv_sum_exp_logits, sum_exp_logits, logits_exp, logits_flat_shifted, logits_flat, logits, 
#            ln3_done, ln3x_hat, ln3var_sqinv, ln3var, ln3diffsq, ln3diff, ln3meani, resconn_mlp_done, resconn_mhsa_done, 
#            mlp_lin2, mlp_drop, mlp_relu, mlp_lin1, ln2_done, ln2x_hat, ln2var_sqinv, ln2var, ln2diffsq, ln2diff, ln2meani, mhsa_done,
#            mhsa_proj, cat_attheads, att3_done, v3, att3_drop, att3, inv_sum_exp_att3, sum_exp_att3, exp_att3, masked_att3_shifted,
#             masked_att3, scaled_dotp3, dotp_att3, k3, q3, att2_done, v2, att2_drop, att2, inv_sum_exp_att2, sum_exp_att2, exp_att2,
#             masked_att2_shifted, masked_att2, scaled_dotp2, dotp_att2, k2, q2, att1_done, v1, att1_drop, att1, inv_sum_exp_att1, 
#             sum_exp_att1, exp_att1, masked_att1_shifted, masked_att1, scaled_dotp1, dotp_att1, k1, q1, ln1_done, ln1x_hat, 
#             ln1var_sqinv, ln1var, ln1diffsq, ln1diff, ln1meani, emb]
# for _tensor in _tensors:
#     _tensor.retain_grad()

# loss.backward()
# loss

In [74]:
# # func to compare my backward pass implementation vs pytorch
# def compare(string, dtensor, _tensor):
#   exact = torch.all(dtensor == _tensor.grad).item()
#   approx = torch.allclose(dtensor, _tensor.grad)
#   maxdiff = (dtensor - _tensor.grad).abs().max().item()
#   print(f'{string:20s} | exact: {str(exact):5s} | approximate: {str(approx):5s} | maxdiff: {maxdiff}')


In [75]:


# #--------------------- cross entropy loss backward pass ------------------------#

# # dloss = 1.0

# dprobs_log = torch.zeros_like(probs_log)
# dprobs_log[torch.arange(len(targets)), targets] = -1.0/len(targets) # * dloss

# dprobs = dprobs_log * 1.0/probs

# #--------------------- softmax backward pass ------------------------#

# dinv_sum_exp_logits = (dprobs * logits_exp).sum(dim=-1, keepdims=True)

# dsum_exp_logits = dinv_sum_exp_logits * -sum_exp_logits**-2

# dlogits_exp = dprobs * inv_sum_exp_logits + dsum_exp_logits

# dlogits_flat_shifted = dlogits_exp * logits_flat_shifted.exp()

# dlogits_flat_temp = torch.zeros_like(logits_flat)
# dlogits_flat_temp[torch.arange(logits_flat.shape[0]), logits_flat.max(dim=-1).indices] = 1
# dlogits_flat = dlogits_flat_shifted - dlogits_flat_shifted.sum(dim=-1, keepdims=True) * dlogits_flat_temp

# #--------------------- flatten backward pass ------------------------#

# dlogits = dlogits_flat.view(B, T, C)

# #--------------------- linear 4 backward pass ------------------------#

# dW4 = dlogits.view(B * T, -1).transpose(-2, -1) @ ln3_done.view(B * T, -1)

# db4 = dlogits.view(B * T, -1).sum(0)

# #--------------------- layer norm 3 backward pass ------------------------#

# dln3_done = dlogits @ W4 

# dgamma3 = (dln3_done * ln3x_hat).view(B * T, -1).sum(0)

# dbeta3 = dln3_done.view(B * T, -1).sum(0)

# dln3x_hat = dln3_done * gamma3

# dln3var_sqinv = (dln3x_hat * ln3diff).sum(dim=-1, keepdims=True)

# dln3var = dln3var_sqinv * -0.5*(ln3var + 1e-5)**-1.5

# dln3diffsq = dln3var * 1/(num_emb_dims - 1) * torch.ones_like(ln3diffsq)

# dln3diff = dln3x_hat * ln3var_sqinv + dln3diffsq * ln3diff * 2

# dln3meani = (dln3diff * -1.0).sum(dim=-1, keepdims=True)

# #--------------------- res conn mlp backward pass ------------------------#

# dresconn_mlp_done = dln3diff + dln3meani * 1/num_emb_dims

# #--------------------- mlp backward pass ------------------------#

# dmlp_lin2 = dresconn_mlp_done

# # linear 3 backward pass 
# dW3 = dmlp_lin2.view(B * T, -1).transpose(-2, -1) @ mlp_drop.view(B * T, -1)

# # linear 3 backward pass 
# db3 = dmlp_lin2.view(B * T, -1).sum(0)

# dmlp_drop = dmlp_lin2 @ W3

# dmlp_relu = dmlp_drop * mlp_dropmask

# dmlp_lin1 = dmlp_relu * (mlp_relu > 0)

# # linear 2 backward pass 
# dW2 = dmlp_lin1.view(B * T, -1).transpose(-2, -1) @ ln2_done.view(B * T, -1)

# # linear 2 backward pass 
# db2 = dmlp_lin1.view(B * T, -1).sum(0)

# #--------------------- layer norm 2 backward pass ------------------------#

# dln2_done = dmlp_lin1 @ W2 

# dgamma2 = (dln2_done * ln2x_hat).view(B * T, -1).sum(0)

# dbeta2 = dln2_done.view(B * T, -1).sum(0)

# dln2x_hat = dln2_done * gamma2

# dln2var_sqinv = (dln2x_hat * ln2diff).sum(dim=-1, keepdims=True)

# dln2var = dln2var_sqinv * -0.5*(ln2var + 1e-5)**-1.5

# dln2diffsq = dln2var * 1/(num_emb_dims - 1) * torch.ones_like(ln2diffsq)

# dln2diff = dln2x_hat * ln2var_sqinv + dln2diffsq * ln2diff * 2

# dln2meani = (dln2diff * -1.0).sum(dim=-1, keepdims=True)

# #--------------------- res conn mhsa backward pass ------------------------#

# dresconn_mhsa_done = dresconn_mlp_done + dln2diff + dln2meani * 1/num_emb_dims

# #--------------------- mhsa backward pass ------------------------#

# dmhsa_done = dresconn_mhsa_done

# dmhsa_proj = dmhsa_done * mhsa_dropmask

# # linear 1 backward pass 
# dW1 = dmhsa_proj.view(B * T, -1).transpose(-2, -1) @ cat_attheads.view(B * T, -1)

# # linear 1 backward pass 
# db1 = dmhsa_proj.view(B * T, -1).sum(0)

# dcat_attheads = dmhsa_proj @ W1

# datt1_done, datt2_done, datt3_done = dcat_attheads.split(head_size, dim=-1)

# #--------------------- shsa3 backward pass ------------------------#

# dv3 = att3_drop.transpose(-2, -1) @ datt3_done 

# datt3_drop = datt3_done @ v3.transpose(-2, -1)

# datt3 = datt3_drop * att3_dropmask

# dinv_sum_exp_att3 = (datt3 * exp_att3).sum(dim=-1, keepdims=True)

# dsum_exp_att3 = dinv_sum_exp_att3* -sum_exp_att3**-2

# dexp_att3 = datt3 * inv_sum_exp_att3 + dsum_exp_att3 * torch.ones_like(exp_att3)

# dmasked_att3_shifted = dexp_att3 * masked_att3_shifted.exp()

# dmasked_att3_temp = torch.zeros_like(masked_att3)
# dmasked_att3_temp.scatter_(-1, masked_att3.max(dim=-1, keepdims=True).indices, 1)
# dmasked_att3 = dmasked_att3_shifted - dmasked_att3_shifted.sum(dim=-1, keepdims=True) * dmasked_att3_temp

# dscaled_dotp3 = dmasked_att3

# ddotp_att3 = dscaled_dotp3 * 1/head_size**0.5

# dk3 = ddotp_att3.transpose(-2, -1) @ q3

# dq3 = ddotp_att3 @ k3

# dWv3 = dv3.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# dWk3 = dk3.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# dWq3 = dq3.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# #--------------------- shsa2 backward pass ------------------------#

# dv2 = att2_drop.transpose(-2, -1) @ datt2_done 

# datt2_drop = datt2_done @ v2.transpose(-2, -1)

# datt2 = datt2_drop * att2_dropmask

# dinv_sum_exp_att2 = (datt2 * exp_att2).sum(dim=-1, keepdims=True)

# dsum_exp_att2 = dinv_sum_exp_att2* -sum_exp_att2**-2

# dexp_att2 = datt2 * inv_sum_exp_att2 + dsum_exp_att2 * torch.ones_like(exp_att2)

# dmasked_att2_shifted = dexp_att2 * masked_att2_shifted.exp()

# dmasked_att2_temp = torch.zeros_like(masked_att2)
# dmasked_att2_temp.scatter_(-1, masked_att2.max(dim=-1, keepdims=True).indices, 1)
# dmasked_att2 = dmasked_att2_shifted - dmasked_att2_shifted.sum(dim=-1, keepdims=True) * dmasked_att2_temp

# dscaled_dotp2 = dmasked_att2

# ddotp_att2 = dscaled_dotp2 * 1/head_size**0.5

# dk2 = ddotp_att2.transpose(-2, -1) @ q2

# dq2 = ddotp_att2 @ k2

# dWv2 = dv2.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# dWk2 = dk2.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# dWq2 = dq2.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# #--------------------- shsa1 backward pass ------------------------#

# dv1 = att1_drop.transpose(-2, -1) @ datt1_done 

# datt1_drop = datt1_done @ v1.transpose(-2, -1)

# datt1 = datt1_drop * att1_dropmask

# dinv_sum_exp_att1 = (datt1 * exp_att1).sum(dim=-1, keepdims=True)

# dsum_exp_att1 = dinv_sum_exp_att1* -sum_exp_att1**-2

# dexp_att1 = datt1 * inv_sum_exp_att1 + dsum_exp_att1 * torch.ones_like(exp_att1)

# dmasked_att1_shifted = dexp_att1 * masked_att1_shifted.exp()

# dmasked_att1_temp = torch.zeros_like(masked_att1)
# dmasked_att1_temp.scatter_(-1, masked_att1.max(dim=-1, keepdims=True).indices, 1)
# dmasked_att1 = dmasked_att1_shifted - dmasked_att1_shifted.sum(dim=-1, keepdims=True) * dmasked_att1_temp

# dscaled_dotp1 = dmasked_att1

# ddotp_att1 = dscaled_dotp1 * 1/head_size**0.5

# dk1 = ddotp_att1.transpose(-2, -1) @ q1

# dq1 = ddotp_att1 @ k1

# dWv1 = dv1.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# dWk1 = dk1.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# dWq1 = dq1.view(B * T, -1).transpose(-2, -1) @ ln1_done.view(B * T, -1)

# #--------------------- layer norm 1 backward pass ------------------------#

# dln1_done = dv3 @ Wv3 + dk3 @ Wk3 + dq3 @ Wq3 + dv2 @ Wv2 + dk2 @ Wk2 + dq2 @ Wq2 + dv1 @ Wv1 + dk1 @ Wk1 + dq1 @ Wq1

# dgamma1 = (dln1_done * ln1x_hat).view(B * T, -1).sum(0)

# dbeta1 = dln1_done.view(B * T, -1).sum(0)

# dln1x_hat = dln1_done * gamma1

# dln1var_sqinv = (dln1x_hat * ln1diff).sum(dim=-1, keepdims=True)

# dln1var = dln1var_sqinv * -0.5*(ln1var + 1e-5)**-1.5

# dln1diffsq = dln1var * 1/(num_emb_dims - 1) * torch.ones_like(ln1diffsq)

# dln1diff = dln1x_hat * ln1var_sqinv + dln1diffsq * ln1diff * 2

# dln1meani = (dln1diff * -1.0).sum(dim=-1, keepdims=True)

# #--------------------- embedding backward pass ------------------------#

# demb = dresconn_mhsa_done + dln1diff + dln1meani * 1/num_emb_dims

# dTOK = torch.zeros_like(TOK)
# for i in range(demb.shape[0]):
#     for j in range(demb.shape[1]):
#         token = xb[i, j]
#         dTOK[token] += demb[i, j]

# dPOS = torch.zeros_like(POS)
# for i in range(demb.shape[0]):
#     for j in range(demb.shape[1]):
#         dPOS[j] += demb[i, j]


In [76]:

# compare("probs_log", dprobs_log, probs_log)
# compare("probs", dprobs, probs)
# compare("inv_sum_exp_logits", dinv_sum_exp_logits, inv_sum_exp_logits)
# compare("sum_exp_logits", dsum_exp_logits, sum_exp_logits)
# compare("logits_exp", dlogits_exp, logits_exp)
# compare("logits_flat_shifted", dlogits_flat_shifted, logits_flat_shifted)
# compare("logits_flat", dlogits_flat, logits_flat)
# compare("logits", dlogits, logits)
# compare("W4", dW4, W4)
# compare("b4", db4, b4)
# compare("ln3_done", dln3_done, ln3_done)
# compare("gamma3", dgamma3, gamma3)
# compare("beta3", dbeta3, beta3)
# compare("ln3x_hat", dln3x_hat, ln3x_hat)
# compare("ln3var_sqinv", dln3var_sqinv, ln3var_sqinv)
# compare("ln3var", dln3var, ln3var)
# compare("ln3diffsq", dln3diffsq, ln3diffsq)
# compare("ln3diff", dln3diff, ln3diff)
# compare("ln3meani", dln3meani, ln3meani)
# compare("resconn_mlp_done", dresconn_mlp_done, resconn_mlp_done)
# compare("resconn_mhsa_done", dresconn_mhsa_done, resconn_mhsa_done)
# compare("mlp_lin2", dmlp_lin2, mlp_lin2)
# compare("W3", dW3, W3)
# compare("b3", db3, b3)
# compare("mlp_drop", dmlp_drop, mlp_drop)
# compare("mlp_relu", dmlp_relu, mlp_relu)
# compare("mlp_lin1", dmlp_lin1, mlp_lin1)
# compare("W2", dW2, W2)
# compare("b2", db2, b2)
# compare("ln2_done", dln2_done, ln2_done)
# compare("gamma2", dgamma2, gamma2)
# compare("beta2", dbeta2, beta2)
# compare("ln2x_hat", dln2x_hat, ln2x_hat)
# compare("ln2var_sqinv", dln2var_sqinv, ln2var_sqinv)
# compare("ln2var", dln2var, ln2var)
# compare("ln2diffsq", dln2diffsq, ln2diffsq)
# compare("ln2diff", dln2diff, ln2diff)
# compare("ln2meani", dln2meani, ln2meani)
# compare("mhsa_done", dmhsa_done, mhsa_done)
# compare("mhsa_proj", dmhsa_proj, mhsa_proj)
# compare("W1", dW1, W1)
# compare("b1", db1, b1)
# compare("cat_attheads", dcat_attheads, cat_attheads)
# compare("att3_done", datt3_done, att3_done)
# compare("v3", dv3, v3)
# compare("att3_drop", datt3_drop, att3_drop)
# compare("att3", datt3, att3)
# compare("inv_sum_exp_att3", dinv_sum_exp_att3, inv_sum_exp_att3)
# compare("sum_exp_att3", dsum_exp_att3, sum_exp_att3)
# compare("exp_att3", dexp_att3, exp_att3)
# compare("masked_att3_shifted", dmasked_att3_shifted, masked_att3_shifted)
# compare("masked_att3", dmasked_att3, masked_att3)
# compare("scaled_dotp3", dscaled_dotp3, scaled_dotp3)
# compare("dotp_att3", ddotp_att3, dotp_att3)
# compare("k3", dk3, k3)
# compare("q3", dq3, q3)
# compare("Wv3", dWv3, Wv3)
# compare("Wk3", dWk3, Wk3)
# compare("Wq3", dWq3, Wq3)
# compare("att2_done", datt2_done, att2_done)
# compare("v2", dv2, v2)
# compare("att2_drop", datt2_drop, att2_drop)
# compare("att2", datt2, att2)
# compare("inv_sum_exp_att2", dinv_sum_exp_att2, inv_sum_exp_att2)
# compare("sum_exp_att2", dsum_exp_att2, sum_exp_att2)
# compare("exp_att2", dexp_att2, exp_att2)
# compare("masked_att2_shifted", dmasked_att2_shifted, masked_att2_shifted)
# compare("masked_att2", dmasked_att2, masked_att2)
# compare("scaled_dotp2", dscaled_dotp2, scaled_dotp2)
# compare("dotp_att2", ddotp_att2, dotp_att2)
# compare("k2", dk2, k2)
# compare("q2", dq2, q2)
# compare("Wv2", dWv2, Wv2)
# compare("Wk2", dWk2, Wk2)
# compare("Wq2", dWq2, Wq2)
# compare("att1_done", datt1_done, att1_done)
# compare("v1", dv1, v1)
# compare("att1_drop", datt1_drop, att1_drop)
# compare("att1", datt1, att1)
# compare("inv_sum_exp_att1", dinv_sum_exp_att1, inv_sum_exp_att1)
# compare("sum_exp_att1", dsum_exp_att1, sum_exp_att1)
# compare("exp_att1", dexp_att1, exp_att1)
# compare("masked_att1_shifted", dmasked_att1_shifted, masked_att1_shifted)
# compare("masked_att1", dmasked_att1, masked_att1)
# compare("scaled_dotp1", dscaled_dotp1, scaled_dotp1)
# compare("dotp_att1", ddotp_att1, dotp_att1)
# compare("k1", dk1, k1)
# compare("q1", dq1, q1)
# compare("Wv1", dWv1, Wv1)
# compare("Wk1", dWk1, Wk1)
# compare("Wq1", dWq1, Wq1)
# compare("ln1_done", dln1_done, ln1_done)
# compare("gamma1", dgamma1, gamma1)
# compare("beta1", dbeta1, beta1)
# compare("ln1x_hat", dln1x_hat, ln1x_hat)
# compare("ln1var_sqinv", dln1var_sqinv, ln1var_sqinv)
# compare("ln1var", dln1var, ln1var)
# compare("ln1diffsq", dln1diffsq, ln1diffsq)
# compare("ln1diff", dln1diff, ln1diff)
# compare("ln1meani", dln1meani, ln1meani)
# compare("emb", demb, emb)
# compare("TOK", dTOK, TOK)
# compare("POS", dPOS, POS)


# --------------------------------------------------------------------------------

### Dataloader

In [77]:

class DataLoader:

    def __init__(self, data, batch_size, context_length, shuffle=True, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.shuffle = shuffle
        self.data = data
        self.batch_size = batch_size
        self.context_length = context_length
        self.total_diff_sequences = len(data) - context_length
        self.num_batches = self.total_diff_sequences // batch_size

    def __iter__(self):
        if self.shuffle:
            idx = torch.randperm(self.total_diff_sequences, generator=self.rng, device=self.device)
        else:
            idx = torch.arange(0, self.total_diff_sequences, device=self.device)
        idx = idx[:self.num_batches * self.batch_size]
        for i in range(0, len(idx), self.batch_size):
            ni = idx[i:i + self.batch_size]
            x = torch.stack([self.data[n : n + self.context_length] for n in ni])
            y = torch.stack([self.data[n + 1 : n + 1 + self.context_length] for n in ni])
            yield x, y
    

### Building blocks

In [ ]:

class Embedding:

    def __init__(self, seq_len, num_emb_dims, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        scale = 0.1/num_emb_dims**0.5
        self.weight = torch.randn((seq_len, num_emb_dims), generator=self.rng, device=self.device) * scale
        self.weight.grad = None

    def __call__(self, idx):
        self.idx = idx
        return self.weight[self.idx]

    def backward(self, dout):
        B, T, C = dout.shape
        self.dW = torch.zeros_like(self.weight, device=self.device)
        idx_flat = self.idx.view(-1)
        dout_flat = dout.view(-1, C)
        self.dW.index_add_(0, idx_flat, dout_flat)
        self.weight.grad = self.dW
        return self.dW

    def parameters(self):
        return [self.weight]

# ------------------------------------------------------------------------------------------------------ #

class LayerNorm:

    def __init__(self, num_emb_dims, eps=1e-5, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.num_emb_dims = num_emb_dims
        self.eps = eps
        self.gamma = torch.randn((num_emb_dims, ), generator=self.rng, device=self.device) * 0.01 + 1.0
        self.beta = torch.randn((num_emb_dims, ), generator=self.rng, device=self.device) * 0.01
        self.gamma.grad = None
        self.beta.grad = None

    def __call__(self, x):
        self.lndiff = x - x.sum(dim=-1, keepdims=True) * 1/self.num_emb_dims
        self.lndiffsq = self.lndiff**2
        self.lnvar = self.lndiffsq.sum(dim=-1, keepdims=True) * 1/(self.num_emb_dims - 1)
        self.lnvar_sqinv = (self.lnvar + 1e-5)**-0.5
        self.lnx_hat = self.lndiff * self.lnvar_sqinv
        ln_done = self.lnx_hat * self.gamma + self.beta
        return ln_done

    def backward(self, dout):
        B, T, C = dout.shape
        self.dgamma = (dout * self.lnx_hat).view(B * T, -1).sum(0)
        self.dbeta = dout.view(B * T, -1).sum(0)
        self.gamma.grad = self.dgamma
        self.beta.grad = self.dbeta
        dlnx_hat = dout * self.gamma
        dlnvar_sqinv = (dlnx_hat * self.lndiff).sum(dim=-1, keepdims=True)
        dlnvar = dlnvar_sqinv * -0.5*(self.lnvar + 1e-5)**-1.5
        dlndiffsq = dlnvar * 1/(self.num_emb_dims - 1) * torch.ones_like(self.lndiffsq, device=self.device)
        dlndiff = dlnx_hat * self.lnvar_sqinv + dlndiffsq * self.lndiff * 2
        dx = dlndiff + (dlndiff * -1.0).sum(dim=-1, keepdims=True) * 1/self.num_emb_dims
        return dx
        

    def parameters(self):
        return [self.gamma] + [self.beta]

# ------------------------------------------------------------------------------------------------------ #

class Linear:

    def __init__(self, features_in, features_out, bias=True, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.features_in = features_in
        self.features_out = features_out     
        scale = 2**0.5 * 1/(self.features_in**0.5) 
        self.weight = torch.randn((self.features_out, self.features_in), generator=self.rng, device=self.device) * scale
        self.weight.grad = None
        if bias:
            self.bias = torch.randn(self.features_out, generator=self.rng, device=self.device) * 0.01
            self.bias.grad = None
        else:
            self.bias = None


    def __call__(self, x):
        self.x = x
        y = self.x @ self.weight.transpose(-2, -1)
        if self.bias is not None:
            y += self.bias
        return y

    def backward(self, dout):
        B, T, C = self.x.shape
        self.dW = dout.view(B * T, -1).transpose(-2, -1) @ self.x.view(B * T, -1)
        self.weight.grad = self.dW
        if self.bias is not None:
            self.db = dout.view(B * T, -1).sum(0)
            self.bias.grad = self.db
        dx = dout @ self.weight
        return dx

    def parameters(self):
        w = [self.weight]
        if self.bias is not None:
            b = [self.bias]
        else:
            b = []
        return w + b

# ------------------------------------------------------------------------------------------------------ #

class SingleHeadSelfAttention:

    def __init__(self,num_emb_dims, head_size, context_length, dropout=0.0, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.att_mask = torch.tril(torch.ones(context_length, context_length, device=self.device))
        self.head_size = head_size
        scale = 2**0.5 * 1/(head_size**0.5)
        self.Wq = torch.randn((self.head_size, num_emb_dims), generator=self.rng, device=self.device) * scale
        self.Wq.grad = None
        self.Wk = torch.randn((self.head_size, num_emb_dims), generator=self.rng, device=self.device) * scale
        self.Wk.grad = None
        self.Wv = torch.randn((self.head_size, num_emb_dims), generator=self.rng, device=self.device) * scale
        self.Wv.grad = None
        self.dropout = Dropout(p=dropout, rng=self.rng, device=self.device)

    def __call__(self, x):
        self.x = x
        self.q = self.x @ self.Wq.transpose(-2, -1)
        self.k = self.x @ self.Wk.transpose(-2, -1)
        self.v = self.x @ self.Wv.transpose(-2, -1)
        dotp_att = self.q @ self.k.transpose(-2, -1)
        scaled_dotp = dotp_att * 1/self.head_size**0.5
        self.masked_att = scaled_dotp.masked_fill(self.att_mask == 0, float("-inf"))
        self.masked_att_shifted = self.masked_att - self.masked_att.max(dim=-1, keepdims=True).values
        self.exp_att = self.masked_att_shifted.exp()
        self.sum_exp_att = self.exp_att.sum(dim=-1, keepdims=True)
        self.inv_sum_exp_att = self.sum_exp_att**-1
        att = self.exp_att * self.inv_sum_exp_att
        self.att_drop = self.dropout(att)  
        att_done = self.att_drop @ self.v
        return att_done    

    def backward(self, dout):
        B, T, C = self.x.shape
        # dout = datt_done
        dv = self.att_drop.transpose(-2, -1) @ dout 
        datt_drop = dout @ self.v.transpose(-2, -1)
        datt = self.dropout.backward(datt_drop)
        dinv_sum_exp_att = (datt * self.exp_att).sum(dim=-1, keepdims=True)
        dsum_exp_att = dinv_sum_exp_att* -self.sum_exp_att**-2
        dexp_att = datt * self.inv_sum_exp_att + dsum_exp_att * torch.ones_like(self.exp_att)
        dmasked_att_shifted = dexp_att * self.masked_att_shifted.exp()
        dmasked_att_temp = torch.zeros_like(self.masked_att, device=self.device)
        dmasked_att_temp.scatter_(-1, self.masked_att.max(dim=-1, keepdims=True).indices, 1)
        dmasked_att = dmasked_att_shifted - dmasked_att_shifted.sum(dim=-1, keepdims=True) * dmasked_att_temp
        dscaled_dotp = dmasked_att
        ddotp_att = dscaled_dotp * 1/self.head_size**0.5
        dk = ddotp_att.transpose(-2, -1) @ self.q
        dq = ddotp_att @ self.k
        self.dWv = dv.view(B * T, -1).transpose(-2, -1) @ self.x.view(B * T, -1)
        self.Wv.grad = self.dWv
        self.dWk = dk.view(B * T, -1).transpose(-2, -1) @ self.x.view(B * T, -1)
        self.Wk.grad = self.dWk
        self.dWq = dq.view(B * T, -1).transpose(-2, -1) @ self.x.view(B * T, -1)
        self.Wq.grad = self.dWq
        return dv @ self.Wv + dk @ self.Wk + dq @ self.Wq
        
    def parameters(self):
        return [self.Wq] + [self.Wk] + [self.Wv]

# ------------------------------------------------------------------------------------------------------ #

class MultiHeadSelfAttention:

    def __init__(self, head_size, num_heads, num_emb_dims, context_length, dropout=0.0, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.head_size = head_size
        self.heads = [
            SingleHeadSelfAttention(num_emb_dims, head_size, context_length, dropout=dropout, rng=self.rng, device=self.device) for _ in range(num_heads)
        ]
        self.Wo = Linear(num_heads*head_size, num_heads*head_size, bias=True, rng=self.rng, device=self.device)
        self.dropout = Dropout(p=dropout, rng=self.rng, device=self.device)

    def __call__(self, x):
        cat_attheads = torch.cat([h(x) for h in self.heads], dim=-1)
        mhsa_proj = self.Wo(cat_attheads)
        return self.dropout(mhsa_proj)

    def backward(self, dout):
        #dmhsa_done = dout (dresconn_mhsa_done)
        dmhsa_proj = self.dropout.backward(dout)
        dcat_attheads = self.Wo.backward(dmhsa_proj)
        datts_done = dcat_attheads.split(self.head_size, dim=-1)
        dx = sum(head.backward(datt_done) for head, datt_done in zip(self.heads, datts_done))
        return dx

    def parameters(self):
        p = []
        for h in self.heads:
            p += h.parameters()
        p +=  self.Wo.parameters()
        return p

# ------------------------------------------------------------------------------------------------------ #

class DecoderBlock:

    def __init__(self, num_emb_dims, num_heads, context_length, dropout=0.0, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        head_size = num_emb_dims // num_heads
        self.ln1 = LayerNorm(num_emb_dims, rng=self.rng, device=self.device)
        self.mhsa = MultiHeadSelfAttention(head_size, num_heads, num_emb_dims, context_length, dropout=dropout, rng=self.rng, device=self.device)
        self.ln2 = LayerNorm(num_emb_dims, rng=self.rng, device=self.device)
        self.mlp = MLP(num_emb_dims, dropout=dropout, rng=self.rng, device=self.device)

    def __call__(self, x):
        x = x + self.mhsa(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

    def backward(self, dout):
        # dout = dresconn_mlp_done
        dln2_done = self.mlp.backward(dout)
        dresconn_mhsa_done = dout + self.ln2.backward(dln2_done)
        dln1_done = self.mhsa.backward(dresconn_mhsa_done)
        dx = dresconn_mhsa_done + self.ln1.backward(dln1_done)
        return dx


    def parameters(self):
        return self.ln1.parameters() + self.mhsa.parameters() + self.ln2.parameters() + self.mlp.parameters()

# ------------------------------------------------------------------------------------------------------ #

class MLP:

    def __init__(self, num_emb_dims, dropout=0.0, hidden_mul=8, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.network = Sequential(
            Linear(num_emb_dims, num_emb_dims * hidden_mul, bias=True, rng=self.rng, device=self.device),
            ReLU(device=self.device),
            Dropout(p=dropout, rng=self.rng, device=self.device),
            Linear(num_emb_dims * hidden_mul, num_emb_dims, bias=True, rng=self.rng, device=self.device),
        )

    def __call__(self, x):
        return self.network(x)

    def backward(self, dout):
        return self.network.backward(dout)

    def parameters(self):
        return self.network.parameters()

# ------------------------------------------------------------------------------------------------------ #

class Sequential:

    def __init__(self, *layers):
        self.layers = list(layers)

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def backward(self, dout):
        for layer in reversed(self.layers):
                dout = layer.backward(dout)
        return dout

    def parameters(self):
        p = []
        for layer in self.layers:
            for param in layer.parameters():
                p.append(param)
        return p

# ------------------------------------------------------------------------------------------------------ #

class Dropout:

    def __init__(self, p=0.0, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.p = p
        self.training = True
        self.mask = None
        
    def __call__(self, x):
        if not self.training or self.p == 0.0:
            return x
        self.mask = (torch.rand(x.shape, generator=self.rng, device=self.device) > self.p) * 1/(1-self.p)  
        return x * self.mask

    def backward(self, dout):
        if self.training and self.p != 0.0:
            dx = dout * self.mask
        else:
            dx = dout
        return dx

    def parameters(self):
        return []

# ------------------------------------------------------------------------------------------------------ #

class ReLU:

    def __init__(self, device='cpu'):
        self.device = device

    def __call__(self, x):
        self.relu = torch.maximum(x, torch.tensor(0, device=self.device))
        return self.relu

    def backward(self, dout):
        dx = dout * (self.relu > 0)
        return dx

    def parameters(self):
        return []

# ------------------------------------------------------------------------------------------------------ #

class Flatten:

    def __call__(self, x, start_dim=0, end_dim=-1):
        dims = len(x.shape)
        start = start_dim if start_dim >= 0 else dims + start_dim
        end = end_dim if end_dim >= 0 else dims + end_dim
        if start > end:
            return x
        new_shape = list(x.shape[:start]) + [-1] + list(x.shape[end + 1:])
        self.initial_shape = x.shape
        return x.reshape(*new_shape)

    def backward(self, dout):
        dx = dout.view(self.initial_shape)
        return dx

    def parameters(self):
        return []

# ------------------------------------------------------------------------------------------------------ #

class Softmax_CrossEntropyLoss:

    def __init__(self, alpha=0.0, device='cpu'):
        self.alpha = alpha
        self.device = device

    def __call__(self, logits_flat, targets, softmax_axis=-1):
        # softmax
        self.logits_flat = logits_flat
        self.logits_flat_shifted = self.logits_flat - self.logits_flat.max(dim=softmax_axis, keepdims=True).values
        self.logits_exp = self.logits_flat_shifted.exp()
        self.sum_exp_logits = self.logits_exp.sum(dim=softmax_axis, keepdims=True)
        self.inv_sum_exp_logits = self.sum_exp_logits**-1
        self.probs = self.logits_exp * self.inv_sum_exp_logits
        # cross entropy loss with label smoothing
        self.targets = targets
        self.vocab_size = self.logits_flat.shape[-1]
        self.batch_size = len(self.targets)
        self.smooth_targets = torch.full((self.batch_size, self.vocab_size), self.alpha / self.vocab_size, device=self.device)
        self.smooth_targets[torch.arange(self.batch_size, device=self.device), self.targets] += (1 - self.alpha)
        self.probs_log = self.probs.log()
        loss = -(self.probs_log * self.smooth_targets).sum(dim=-1).mean()
        return loss

    def backward(self, dout):
        # dout = 1.0 (dloss)
        dprobs_log = torch.zeros_like(self.probs_log, device=self.device)
        dprobs_log = -self.smooth_targets / self.batch_size
        dprobs = dprobs_log * 1.0/self.probs
        dinv_sum_exp_logits = (dprobs * self.logits_exp).sum(dim=-1, keepdims=True)
        dsum_exp_logits = dinv_sum_exp_logits * -self.sum_exp_logits**-2
        dlogits_exp = dprobs * self.inv_sum_exp_logits + dsum_exp_logits
        dlogits_flat_shifted = dlogits_exp * self.logits_flat_shifted.exp() 
        dlogits_flat_temp = torch.zeros_like(self.logits_flat, device=self.device)
        dlogits_flat_temp[torch.arange(self.logits_flat.shape[0], device=self.device), self.logits_flat.max(dim=-1).indices] = 1
        dlogits_flat = dlogits_flat_shifted - dlogits_flat_shifted.sum(dim=-1, keepdims=True) * dlogits_flat_temp
        return dlogits_flat
        

    def parameters(self):
        return []



### Model

In [79]:


class Model:

    def __init__(self, vocab_size, num_emb_dims, context_length, num_heads, num_layers, dropout=0.0, alpha=0.0, rng=None, device='cpu'):
        self.device = device
        self.rng = rng
        self.token_embeddings = Embedding(vocab_size, num_emb_dims, rng=self.rng, device=self.device)
        self.position_embeddings = Embedding(context_length, num_emb_dims, rng=self.rng, device=self.device)
        self.decoder_blocks = Sequential(
            *[DecoderBlock(num_emb_dims, num_heads, context_length, dropout=dropout, rng=self.rng, device=self.device) for _ in range(num_layers)]
        )
        self.ln = LayerNorm(num_emb_dims, rng=self.rng, device=self.device)
        self.model_head = Linear(num_emb_dims, vocab_size, bias=True, rng=self.rng, device=self.device)
        self.flatten_logits = Flatten()
        self.flatten_targets = Flatten()
        self.soft_ce_loss = Softmax_CrossEntropyLoss(alpha=alpha, device=self.device)

    def __call__(self, indexes, targets=None):
        self.indexes = indexes
        B, T = self.indexes.shape
        te = self.token_embeddings(self.indexes)
        pe = self.position_embeddings(torch.arange(T, device=self.device).repeat(B, 1))
        x =  te + pe
        x = self.decoder_blocks(x)
        x = self.ln(x)
        logits = self.model_head(x)
        if targets is None:
            loss = None
        else:
            logits_flat = self.flatten_logits(logits, end_dim=1)
            targets_flat = self.flatten_targets(targets)            
            loss = self.soft_ce_loss(logits_flat, targets_flat, softmax_axis=-1)
        return logits, loss

    def backward(self, dloss):
        dlogits_flat = self.soft_ce_loss.backward(dloss)
        dlogits = self.flatten_logits.backward(dlogits_flat)
        dln_done = self.model_head.backward(dlogits)
        dresconn_mlp_done = self.ln.backward(dln_done)
        demb = self.decoder_blocks.backward(dresconn_mlp_done)
        dTOK = self.token_embeddings.backward(demb)
        dPOS = self.position_embeddings.backward(demb)

    def zero_grad(self):
        for p in self.parameters():
            p.grad = None
            
    def parameters(self):
        return self.token_embeddings.parameters() + self.position_embeddings.parameters() \
                + self.decoder_blocks.parameters() + self.ln.parameters() + self.model_head.parameters()


### Grad helper func

In [80]:

def get_params_and_grads(model):
    params_and_grads = []
    for p in model.parameters():
        params_and_grads.append((p, p.grad))
    return params_and_grads



### Optimizer

In [81]:

class AdamW:
    def __init__(self, max_lr=1e-3, min_lr=1e-4, warmup_steps=1000, max_steps=10000, beta1=0.9, beta2=0.95, eps=1e-8, weight_decay=0.0, max_norm=1.0):
        self.max_lr = max_lr
        self.min_lr = min_lr
        self.warmup_steps = warmup_steps
        self.max_steps = max_steps
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.max_norm = max_norm
        self.step_count = 0
        self.m = {}
        self.v = {}

    def step(self, params_and_grads):
        self.step_count += 1

        params, grads = zip(*params_and_grads)
        grads = self.grad_clipping(grads)
        pgs = zip(params, grads)

        for param, grad in pgs:
            if grad is None:
                continue

            param_id = id(param)
            if param_id not in self.m:
                self.m[param_id] = torch.zeros_like(param)
                self.v[param_id] = torch.zeros_like(param)

            self.m[param_id] = self.beta1 * self.m[param_id] + (1 - self.beta1) * grad
            self.v[param_id] = self.beta2 * self.v[param_id] + (1 - self.beta2) * grad**2
            bias_correction_m = 1 - self.beta1**self.step_count
            bias_correction_v = 1 - self.beta2**self.step_count
            m_corrected = self.m[param_id] / bias_correction_m
            v_corrected = self.v[param_id] / bias_correction_v

            param -= self.get_lr() * (m_corrected / (v_corrected + self.eps)**0.5 + self.weight_decay * param)

    def grad_clipping(self, grads):
        total_norm = 0.0
        for grad in grads:
            if grad is not None:
                total_norm += torch.linalg.norm(grad)**2
        total_norm = total_norm**0.5

        if total_norm > self.max_norm:
            coeff = self.max_norm / total_norm
            clipped_grads = [grad * coeff for grad in grads]
            return clipped_grads
        else:
            return grads

    def get_lr(self):
        if self.step_count < self.warmup_steps:
            return self.max_lr * ((self.step_count + 1) / self.warmup_steps)
        elif self.step_count < self.max_steps:
            # cosine decay
            progress = (self.step_count - self.warmup_steps) / (self.max_steps - self.warmup_steps)
            return self.min_lr + (self.max_lr - self.min_lr) * (1 + math.cos(math.pi * progress)) / 2
        else:
            return self.min_lr
        





# ------------------------------------------------------------------------------

### Configs

In [ ]:

@dataclass
class Configs:
    vocab_size: int = voc_size

    batch_size: int = 64         # B
    context_length: int = 256    # T
    num_emb_dims: int = 320      # C

    num_heads: int = 5
    num_layers: int = 3

    dropout: float = 0.15
    max_lr: float = 1e-3
    min_lr: float = 1e-5
    warmup_steps: int = 2500
    max_steps: int = 75000
    weight_decay: float = 0.30
    alpha: float = 0.0


### Device + Model

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

In [84]:
rng = torch.Generator(device=device).manual_seed(39)
cfg = Configs()
model = Model(cfg.vocab_size, cfg.num_emb_dims, cfg.context_length, cfg.num_heads, cfg.num_layers, dropout=cfg.dropout, alpha=cfg.alpha, rng=rng, device=device)

### Parameters count

In [ ]:
print(sum(p.numel() for p in model.parameters()))

### Training

In [ ]:


train_batches = DataLoader(train_data, cfg.batch_size, cfg.context_length, shuffle=True, rng=rng, device=device)
opt = AdamW(
            max_lr=cfg.max_lr,
            min_lr=cfg.min_lr,
            warmup_steps=cfg.warmup_steps,
            max_steps=cfg.max_steps,
            weight_decay=cfg.weight_decay,
)   

steps = 0
train_loss = []
val_loss = []
val_steps = []

for x, y in train_batches:
    x, y = x.to(device), y.to(device)

    # forward pass
    _logits, t_loss = model(x, y)

    # zero out grads
    model.zero_grad()
    # backward pass
    model.backward(1.0)

    # update weights
    pgs = get_params_and_grads(model)
    opt.step(pgs)

    steps += 1
    train_loss.append(t_loss.item())


    if steps % 500 == 0:
        val_batches = DataLoader(val_data, cfg.batch_size, cfg.context_length, shuffle=True, rng=rng, device=device)
        num_val_loss = 50
        total_v_loss = 0

        for i, (xv, yv) in enumerate(val_batches):
            if i >= num_val_loss:
                break
            xv, yv = xv.to(device), yv.to(device)

            _, v_loss = model(xv, yv)
            total_v_loss += v_loss.item()

        avg_v_loss = total_v_loss / num_val_loss
        val_loss.append(avg_v_loss)
        val_steps.append(steps)

        print(f"step: {steps} | tr_loss: {t_loss.item()} | val_loss: {avg_v_loss}")
        print("----------------------------------------------------------------------------")

        


### Loss

In [ ]:

plt.plot(torch.arange(steps), train_loss)
plt.title('Training loss')
plt.xlabel("steps")
plt.ylabel("loss")
plt.show()

In [ ]:

plt.plot(val_steps, val_loss)
plt.title('Val loss')
plt.xlabel("steps")
plt.ylabel("loss")
plt.show()

### Accuracy

In [25]:
def calculate_accuracy(logits, targets):
    predictions = torch.argmax(logits, dim=-1)
    correct = (predictions == targets).float()
    return correct.mean().item()

def evaluate(model, val_data, cfg, device, rng):
    val_batches = DataLoader(val_data, cfg.batch_size, cfg.context_length, rng=rng, device=device)
    val_loss = 0
    total_accuracy = 0

    for x, y in val_batches:
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        val_loss += loss.item()
        total_accuracy += calculate_accuracy(logits, y)
    
    total_diff_sequences = len(val_data) - cfg.context_length
    num_batches = total_diff_sequences // cfg.batch_size

    return val_loss / num_batches, total_accuracy / num_batches




In [ ]:
val_loss, val_accuracy = evaluate(model, val_data, cfg, device, rng)
print(f"Validation Loss: {val_loss}, Validation Accuracy: {val_accuracy}")

### Generate text from model

In [28]:

def generate(model, indexes, max_new_tokens):

    for i in range(max_new_tokens):
        logits, _ = model(indexes[:, i:])

        # get last char
        logits = logits[:, -1, :] 
        # softmax
        logits_shifted = logits - logits.max(dim=-1, keepdims=True).values
        probs = logits_shifted.exp() / logits_shifted.exp().sum(dim=-1, keepdims=True)
        # sample
        next_index = torch.multinomial(probs, num_samples=1)
        # concat
        indexes = torch.cat((indexes, next_index), dim=-1)
        
    return indexes

In [ ]:
indexes = torch.zeros((1, cfg.context_length),  dtype=torch.long, device=device)

generated_indexes = generate(model, indexes, 20000)

In [ ]:
print(decode(generated_indexes[0, 256:].tolist(), inv_vocab))

### Best results

In [ ]:

# model params count: 6298972, dataset size: 6m with 90-10 split

# Validation Loss: 0.6895036889857714, Validation Accuracy: 0.7852043564629242

# step: 500 | tr_loss: 2.411581516265869 | val_loss: 2.3716541576385497
# ----------------------------------------------------------------------------
# step: 1000 | tr_loss: 2.1743340492248535 | val_loss: 2.1475356578826905
# ----------------------------------------------------------------------------
# step: 1500 | tr_loss: 1.6932158470153809 | val_loss: 1.6793347692489624
# ----------------------------------------------------------------------------
# step: 2000 | tr_loss: 1.4505586624145508 | val_loss: 1.4652190327644348
# ----------------------------------------------------------------------------
# step: 2500 | tr_loss: 1.2270088195800781 | val_loss: 1.287634210586548
# ----------------------------------------------------------------------------
# step: 3000 | tr_loss: 1.166236162185669 | val_loss: 1.163523781299591
# ----------------------------------------------------------------------------
# step: 3500 | tr_loss: 1.049368977546692 | val_loss: 1.0737783050537109
# ----------------------------------------------------------------------------
# step: 4000 | tr_loss: 1.0197346210479736 | val_loss: 1.0145488965511322
# ----------------------------------------------------------------------------
# step: 4500 | tr_loss: 0.9447590112686157 | val_loss: 0.9672004151344299
# ----------------------------------------------------------------------------
# step: 5000 | tr_loss: 0.9027822017669678 | val_loss: 0.9345887839794159
# ----------------------------------------------------------------------------
# step: 5500 | tr_loss: 0.8991243839263916 | val_loss: 0.9160502207279205
# ----------------------------------------------------------------------------
# step: 6000 | tr_loss: 0.8740971088409424 | val_loss: 0.8940156376361847
# ----------------------------------------------------------------------------
# step: 6500 | tr_loss: 0.8640769720077515 | val_loss: 0.875804010629654
# ----------------------------------------------------------------------------
# step: 7000 | tr_loss: 0.8283149003982544 | val_loss: 0.8667150008678436
# ----------------------------------------------------------------------------
# step: 7500 | tr_loss: 0.8298428058624268 | val_loss: 0.8538200914859772
# ----------------------------------------------------------------------------
# step: 8000 | tr_loss: 0.774289608001709 | val_loss: 0.8531646990776062
# ----------------------------------------------------------------------------
# step: 8500 | tr_loss: 0.7941063642501831 | val_loss: 0.834422756433487
# ----------------------------------------------------------------------------
# step: 9000 | tr_loss: 0.8150866031646729 | val_loss: 0.8338386476039886
# ----------------------------------------------------------------------------
# step: 9500 | tr_loss: 0.8145318627357483 | val_loss: 0.8261837267875671
# ----------------------------------------------------------------------------
# step: 10000 | tr_loss: 0.785689651966095 | val_loss: 0.8190973091125489
# ----------------------------------------------------------------------------
# step: 10500 | tr_loss: 0.7993773221969604 | val_loss: 0.8161746394634247
# ----------------------------------------------------------------------------
# step: 11000 | tr_loss: 0.8111218214035034 | val_loss: 0.8086070692539216
# ----------------------------------------------------------------------------
# step: 11500 | tr_loss: 0.7524799704551697 | val_loss: 0.8025170958042145
# ----------------------------------------------------------------------------
# step: 12000 | tr_loss: 0.7530072927474976 | val_loss: 0.7950506031513214
# ----------------------------------------------------------------------------
# step: 12500 | tr_loss: 0.7761865854263306 | val_loss: 0.7964671933650971
# ----------------------------------------------------------------------------
# step: 13000 | tr_loss: 0.7937381267547607 | val_loss: 0.7964484989643097
# ----------------------------------------------------------------------------
# step: 13500 | tr_loss: 0.7607235908508301 | val_loss: 0.7910967338085174
# ----------------------------------------------------------------------------
# step: 14000 | tr_loss: 0.7858783006668091 | val_loss: 0.7909491550922394
# ----------------------------------------------------------------------------
# step: 14500 | tr_loss: 0.7608457803726196 | val_loss: 0.7893683087825775
# ----------------------------------------------------------------------------
# step: 15000 | tr_loss: 0.7409508228302002 | val_loss: 0.7845044362545014
# ----------------------------------------------------------------------------
# step: 15500 | tr_loss: 0.752314031124115 | val_loss: 0.7803300762176514
# ----------------------------------------------------------------------------
# step: 16000 | tr_loss: 0.7574610710144043 | val_loss: 0.7849794089794159
# ----------------------------------------------------------------------------
# step: 16500 | tr_loss: 0.7578995227813721 | val_loss: 0.781592127084732
# ----------------------------------------------------------------------------
# step: 17000 | tr_loss: 0.7441802024841309 | val_loss: 0.7763706409931183
# ----------------------------------------------------------------------------
# step: 17500 | tr_loss: 0.7491922974586487 | val_loss: 0.7819801163673401
# ----------------------------------------------------------------------------
# step: 18000 | tr_loss: 0.7709373235702515 | val_loss: 0.7750989758968353
# ----------------------------------------------------------------------------
# step: 18500 | tr_loss: 0.732083261013031 | val_loss: 0.7754498589038848
# ----------------------------------------------------------------------------
# step: 19000 | tr_loss: 0.7231558561325073 | val_loss: 0.7754987061023713
# ----------------------------------------------------------------------------
# step: 19500 | tr_loss: 0.7211115956306458 | val_loss: 0.7726799881458283
# ----------------------------------------------------------------------------
# step: 20000 | tr_loss: 0.7419028282165527 | val_loss: 0.7750193977355957
# ----------------------------------------------------------------------------
# step: 20500 | tr_loss: 0.7092651128768921 | val_loss: 0.7695248568058014
# ----------------------------------------------------------------------------
# step: 21000 | tr_loss: 0.7158918380737305 | val_loss: 0.7698352122306824
# ----------------------------------------------------------------------------
# step: 21500 | tr_loss: 0.7367719411849976 | val_loss: 0.7673097729682923
# ----------------------------------------------------------------------------
# step: 22000 | tr_loss: 0.7413603067398071 | val_loss: 0.7622463428974151
# ----------------------------------------------------------------------------
# step: 22500 | tr_loss: 0.7152903079986572 | val_loss: 0.7676921463012696
# ----------------------------------------------------------------------------
# step: 23000 | tr_loss: 0.7306903600692749 | val_loss: 0.76648108959198
# ----------------------------------------------------------------------------
# step: 23500 | tr_loss: 0.7330034375190735 | val_loss: 0.7627314043045044
# ----------------------------------------------------------------------------
# step: 24000 | tr_loss: 0.7045186161994934 | val_loss: 0.760528529882431
# ----------------------------------------------------------------------------
# step: 24500 | tr_loss: 0.6805546283721924 | val_loss: 0.7525082325935364
# ----------------------------------------------------------------------------
# step: 25000 | tr_loss: 0.7099364995956421 | val_loss: 0.7567629134654998
# ----------------------------------------------------------------------------
# step: 25500 | tr_loss: 0.7150777578353882 | val_loss: 0.7602011168003082
# ----------------------------------------------------------------------------
# step: 26000 | tr_loss: 0.7116530537605286 | val_loss: 0.7549095261096954
# ----------------------------------------------------------------------------
# step: 26500 | tr_loss: 0.7048287391662598 | val_loss: 0.7527936732769013
# ----------------------------------------------------------------------------
# step: 27000 | tr_loss: 0.734254002571106 | val_loss: 0.7542327117919921
# ----------------------------------------------------------------------------
# step: 27500 | tr_loss: 0.7023987174034119 | val_loss: 0.7475447368621826
# ----------------------------------------------------------------------------
# step: 28000 | tr_loss: 0.734488844871521 | val_loss: 0.747799322605133
# ----------------------------------------------------------------------------
# step: 28500 | tr_loss: 0.715897262096405 | val_loss: 0.7464596343040466
# ----------------------------------------------------------------------------
# step: 29000 | tr_loss: 0.7014802098274231 | val_loss: 0.7531375956535339
# ----------------------------------------------------------------------------
# step: 29500 | tr_loss: 0.7177521586418152 | val_loss: 0.7457726550102234
# ----------------------------------------------------------------------------
# step: 30000 | tr_loss: 0.6826249957084656 | val_loss: 0.7499308252334594
# ----------------------------------------------------------------------------
# step: 30500 | tr_loss: 0.7042731046676636 | val_loss: 0.7443495094776154
# ----------------------------------------------------------------------------
# step: 31000 | tr_loss: 0.7047593593597412 | val_loss: 0.7453215801715851
# ----------------------------------------------------------------------------
# step: 31500 | tr_loss: 0.6857190132141113 | val_loss: 0.7451656043529511
# ----------------------------------------------------------------------------
# step: 32000 | tr_loss: 0.7055763006210327 | val_loss: 0.743225269317627
# ----------------------------------------------------------------------------
# step: 32500 | tr_loss: 0.7046209573745728 | val_loss: 0.7393017911911011
# ----------------------------------------------------------------------------
# step: 33000 | tr_loss: 0.7122840881347656 | val_loss: 0.7398750448226928
# ----------------------------------------------------------------------------
# step: 33500 | tr_loss: 0.7057012319564819 | val_loss: 0.7388281977176666
# ----------------------------------------------------------------------------
# step: 34000 | tr_loss: 0.6896214485168457 | val_loss: 0.7434128320217133
# ----------------------------------------------------------------------------
# step: 34500 | tr_loss: 0.6921489238739014 | val_loss: 0.7389706301689148
# ----------------------------------------------------------------------------
# step: 35000 | tr_loss: 0.7014720439910889 | val_loss: 0.7372801303863525
# ----------------------------------------------------------------------------
# step: 35500 | tr_loss: 0.6753056049346924 | val_loss: 0.7384328269958496
# ----------------------------------------------------------------------------
# step: 36000 | tr_loss: 0.6645851135253906 | val_loss: 0.7355722892284393
# ----------------------------------------------------------------------------
# step: 36500 | tr_loss: 0.6902304887771606 | val_loss: 0.7328090870380402
# ----------------------------------------------------------------------------
# step: 37000 | tr_loss: 0.7017063498497009 | val_loss: 0.7363377737998963
# ----------------------------------------------------------------------------
# step: 37500 | tr_loss: 0.6791359186172485 | val_loss: 0.7330827248096466
# ----------------------------------------------------------------------------
# step: 38000 | tr_loss: 0.6541965007781982 | val_loss: 0.7311620271205902
# ----------------------------------------------------------------------------
# step: 38500 | tr_loss: 0.6778802871704102 | val_loss: 0.7309894251823426
# ----------------------------------------------------------------------------
# step: 39000 | tr_loss: 0.6711705923080444 | val_loss: 0.7242380368709564
# ----------------------------------------------------------------------------
# step: 39500 | tr_loss: 0.6638603806495667 | val_loss: 0.7283755469322205
# ----------------------------------------------------------------------------
# step: 40000 | tr_loss: 0.6824849247932434 | val_loss: 0.7249944031238555
# ----------------------------------------------------------------------------
# step: 40500 | tr_loss: 0.6755020618438721 | val_loss: 0.7264903473854065
# ----------------------------------------------------------------------------
# step: 41000 | tr_loss: 0.662254810333252 | val_loss: 0.7215487432479858
# ----------------------------------------------------------------------------
# step: 41500 | tr_loss: 0.6657676696777344 | val_loss: 0.7242959523200989
# ----------------------------------------------------------------------------
# step: 42000 | tr_loss: 0.6582472324371338 | val_loss: 0.7233211660385132
# ----------------------------------------------------------------------------
# step: 42500 | tr_loss: 0.6320792436599731 | val_loss: 0.7204104065895081
# ----------------------------------------------------------------------------
# step: 43000 | tr_loss: 0.6517252922058105 | val_loss: 0.7185541474819184
# ----------------------------------------------------------------------------
# step: 43500 | tr_loss: 0.6283149719238281 | val_loss: 0.7249444675445557
# ----------------------------------------------------------------------------
# step: 44000 | tr_loss: 0.6961092352867126 | val_loss: 0.7233854281902313
# ----------------------------------------------------------------------------
# step: 44500 | tr_loss: 0.662893533706665 | val_loss: 0.7194882476329804
# ----------------------------------------------------------------------------
# step: 45000 | tr_loss: 0.6520633697509766 | val_loss: 0.7142968690395355
# ----------------------------------------------------------------------------
# step: 45500 | tr_loss: 0.6559734344482422 | val_loss: 0.7255239188671112
# ----------------------------------------------------------------------------
# step: 46000 | tr_loss: 0.6537927389144897 | val_loss: 0.7136680901050567
# ----------------------------------------------------------------------------
# step: 46500 | tr_loss: 0.6443434357643127 | val_loss: 0.7178300797939301
# ----------------------------------------------------------------------------
# step: 47000 | tr_loss: 0.6586759686470032 | val_loss: 0.7152275228500367
# ----------------------------------------------------------------------------
# step: 47500 | tr_loss: 0.6081377863883972 | val_loss: 0.712113254070282
# ----------------------------------------------------------------------------
# step: 48000 | tr_loss: 0.6060612201690674 | val_loss: 0.7099326276779174
# ----------------------------------------------------------------------------
# step: 48500 | tr_loss: 0.6495673656463623 | val_loss: 0.7106005454063415
# ----------------------------------------------------------------------------
# step: 49000 | tr_loss: 0.6641571521759033 | val_loss: 0.7095865416526794
# ----------------------------------------------------------------------------
# step: 49500 | tr_loss: 0.6883050799369812 | val_loss: 0.7129636609554291
# ----------------------------------------------------------------------------
# step: 50000 | tr_loss: 0.628922700881958 | val_loss: 0.7132633030414581
# ----------------------------------------------------------------------------
# step: 50500 | tr_loss: 0.6369253396987915 | val_loss: 0.7065528404712677
# ----------------------------------------------------------------------------
# step: 51000 | tr_loss: 0.6274491548538208 | val_loss: 0.7110602247714997
# ----------------------------------------------------------------------------
# step: 51500 | tr_loss: 0.6331945657730103 | val_loss: 0.7064776957035065
# ----------------------------------------------------------------------------
# step: 52000 | tr_loss: 0.6308190822601318 | val_loss: 0.7081779956817627
# ----------------------------------------------------------------------------
# step: 52500 | tr_loss: 0.6278111934661865 | val_loss: 0.7057689023017883
# ----------------------------------------------------------------------------
# step: 53000 | tr_loss: 0.637445330619812 | val_loss: 0.7002361106872559
# ----------------------------------------------------------------------------
# step: 53500 | tr_loss: 0.6028193235397339 | val_loss: 0.706443532705307
# ----------------------------------------------------------------------------
# step: 54000 | tr_loss: 0.6514031887054443 | val_loss: 0.7034680628776551
# ----------------------------------------------------------------------------
# step: 54500 | tr_loss: 0.626995325088501 | val_loss: 0.6988164067268372
# ----------------------------------------------------------------------------
# step: 55000 | tr_loss: 0.6262666583061218 | val_loss: 0.6994020128250122
# ----------------------------------------------------------------------------
# step: 55500 | tr_loss: 0.6345035433769226 | val_loss: 0.7029903709888459
# ----------------------------------------------------------------------------
# step: 56000 | tr_loss: 0.6224568486213684 | val_loss: 0.6988896572589874
# ----------------------------------------------------------------------------
# step: 56500 | tr_loss: 0.5911512970924377 | val_loss: 0.6983276855945587
# ----------------------------------------------------------------------------
# step: 57000 | tr_loss: 0.6117769479751587 | val_loss: 0.6979424905776977
# ----------------------------------------------------------------------------
# step: 57500 | tr_loss: 0.5724279880523682 | val_loss: 0.6980634236335754
# ----------------------------------------------------------------------------
# step: 58000 | tr_loss: 0.6110518574714661 | val_loss: 0.6929619896411896
# ----------------------------------------------------------------------------
# step: 58500 | tr_loss: 0.5986217260360718 | val_loss: 0.6988119995594024
# ----------------------------------------------------------------------------
# step: 59000 | tr_loss: 0.6046183109283447 | val_loss: 0.698676677942276
# ----------------------------------------------------------------------------
# step: 59500 | tr_loss: 0.6238846778869629 | val_loss: 0.6948799657821655
# ----------------------------------------------------------------------------
# step: 60000 | tr_loss: 0.6190516948699951 | val_loss: 0.7020801782608033
# ----------------------------------------------------------------------------
# step: 60500 | tr_loss: 0.595536470413208 | val_loss: 0.6995361399650574
# ----------------------------------------------------------------------------
# step: 61000 | tr_loss: 0.606037437915802 | val_loss: 0.6976299345493316
# ----------------------------------------------------------------------------
# step: 61500 | tr_loss: 0.5701172351837158 | val_loss: 0.6942230916023254
# ----------------------------------------------------------------------------
# step: 62000 | tr_loss: 0.5905520915985107 | val_loss: 0.694314022064209
# ----------------------------------------------------------------------------
# step: 62500 | tr_loss: 0.5847295522689819 | val_loss: 0.6945355319976807
# ----------------------------------------------------------------------------
# step: 63000 | tr_loss: 0.592321515083313 | val_loss: 0.6933014142513275
# ----------------------------------------------------------------------------
# step: 63500 | tr_loss: 0.5783157348632812 | val_loss: 0.6937134933471679
# ----------------------------------------------------------------------------
# step: 64000 | tr_loss: 0.5853142142295837 | val_loss: 0.6955509042739868
# ----------------------------------------------------------------------------
# step: 64500 | tr_loss: 0.5642961263656616 | val_loss: 0.693478319644928
# ----------------------------------------------------------------------------
# step: 65000 | tr_loss: 0.5866572856903076 | val_loss: 0.6975683522224426
# ----------------------------------------------------------------------------
# step: 65500 | tr_loss: 0.5819246768951416 | val_loss: 0.692070163488388
# ----------------------------------------------------------------------------
# step: 66000 | tr_loss: 0.5827187299728394 | val_loss: 0.6900880086421967
# ----------------------------------------------------------------------------
# step: 66500 | tr_loss: 0.561967134475708 | val_loss: 0.6905195748806
# ----------------------------------------------------------------------------
# step: 67000 | tr_loss: 0.5844035148620605 | val_loss: 0.689392158985138
# ----------------------------------------------------------------------------
# step: 67500 | tr_loss: 0.5726150274276733 | val_loss: 0.6884672009944915
# ----------------------------------------------------------------------------
# step: 68000 | tr_loss: 0.5725446939468384 | val_loss: 0.6924813568592072
# ----------------------------------------------------------------------------
# step: 68500 | tr_loss: 0.5847375392913818 | val_loss: 0.6878006184101104
# ----------------------------------------------------------------------------
# step: 69000 | tr_loss: 0.5712884664535522 | val_loss: 0.690440970659256
# ----------------------------------------------------------------------------
# step: 69500 | tr_loss: 0.5672208070755005 | val_loss: 0.6909021496772766
# ----------------------------------------------------------------------------
# step: 70000 | tr_loss: 0.5682793259620667 | val_loss: 0.6898853075504303
# ----------------------------------------------------------------------------
# step: 70500 | tr_loss: 0.5589553117752075 | val_loss: 0.6853524601459503
# ----------------------------------------------------------------------------
# step: 71000 | tr_loss: 0.5766798257827759 | val_loss: 0.6892390036582947
# ----------------------------------------------------------------------------
# step: 71500 | tr_loss: 0.5704895257949829 | val_loss: 0.6850900971889495
# ----------------------------------------------------------------------------
# step: 72000 | tr_loss: 0.56464022397995 | val_loss: 0.6911041283607483
# ----------------------------------------------------------------------------
# step: 72500 | tr_loss: 0.5753606557846069 | val_loss: 0.6855942678451538
# ----------------------------------------------------------------------------
# step: 73000 | tr_loss: 0.5564370155334473 | val_loss: 0.6853385758399964
# ----------------------------------------------------------------------------
# step: 73500 | tr_loss: 0.5374496579170227 | val_loss: 0.6853627753257752
# ----------------------------------------------------------------------------
# step: 74000 | tr_loss: 0.5707399845123291 | val_loss: 0.6893702268600463
# ----------------------------------------------------------------------------
# step: 74500 | tr_loss: 0.561547577381134 | val_loss: 0.6901405680179596
# ----------------------------------------------------------------------------
# step: 75000 | tr_loss: 0.5584772825241089 | val_loss: 0.6908338701725006
# ----------------------------------------------------------------------------
# step: 75500 | tr_loss: 0.5670167207717896 | val_loss: 0.6858243060111999
# ----------------------------------------------------------------------------
# step: 76000 | tr_loss: 0.5630859136581421 | val_loss: 0.6915340828895569
# ----------------------------------------------------------------------------
# step: 76500 | tr_loss: 0.56833416223526 | val_loss: 0.6873970592021942
# ----------------------------------------------------------------------------
# step: 77000 | tr_loss: 0.5706148743629456 | val_loss: 0.6863424146175384
# ----------------------------------------------------------------------------
# step: 77500 | tr_loss: 0.5701026916503906 | val_loss: 0.6896999859809876
# ----------------------------------------------------------------------------
# step: 78000 | tr_loss: 0.5576578378677368 | val_loss: 0.6884762418270111
# ----------------------------------------------------------------------------
# step: 78500 | tr_loss: 0.5617331266403198 | val_loss: 0.6874490034580231
# ----------------------------------------------------------------------------
# step: 79000 | tr_loss: 0.5359886288642883 | val_loss: 0.6910885906219483
# ----------------------------------------------------------------------------
# step: 79500 | tr_loss: 0.5416364669799805 | val_loss: 0.6884203946590424
# ----------------------------------------------------------------------------
# step: 80000 | tr_loss: 0.5578609108924866 | val_loss: 0.6894981753826142
# ----------------------------------------------------------------------------
# step: 80500 | tr_loss: 0.5558005571365356 | val_loss: 0.6845698249340058
# ----------------------------------------------------------------------------
# step: 81000 | tr_loss: 0.5764395594596863 | val_loss: 0.6873912990093232
# ----------------------------------------------------------------------------
# step: 81500 | tr_loss: 0.5710822939872742 | val_loss: 0.6917544615268707
# ----------------------------------------------------------------------------
# step: 82000 | tr_loss: 0.5831484794616699 | val_loss: 0.6857198202610015
# ----------------------------------------------------------------------------
# step: 82500 | tr_loss: 0.5743141174316406 | val_loss: 0.6858723282814025
# ----------------------------------------------------------------------------
# step: 83000 | tr_loss: 0.5469778180122375 | val_loss: 0.6910465943813324
# ----------------------------------------------------------------------------
# step: 83500 | tr_loss: 0.5510320067405701 | val_loss: 0.692688672542572
# ----------------------------------------------------------------------------
# step: 84000 | tr_loss: 0.5852558612823486 | val_loss: 0.6869754636287689
# ----------------------------------------------------------------------------